# LexFrame quickstart

LexFrame reads a hidden state the way the model reads it, in the model's own vocabulary: what the state carries by itself, how the model's output responds to it, who wrote it, and how moving it changes the prediction. This notebook reads SmolLM3-3B, compares LexFrame with the logit lens and the Jacobian lens, and steers a prediction. Choose a GPU runtime (Runtime > Change runtime type > T4 GPU).

[Paper](https://arxiv.org/abs/2610.22169) | [Blog post](https://jianguda.github.io/blogs/lexframe/) | [Code](https://github.com/jianguda/lexframe)

In [ ]:
!pip install -q lexframe datasets matplotlib

## Load a model and calibrate a frame

Any causal language model in Hugging Face Transformers works. The frame is calibrated once on unlabelled text, 1024 blocks spread over WikiText-103, which takes a few minutes and is cached on disk.

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

import lexframe as lf

name = "HuggingFaceTB/SmolLM3-3B"
model = AutoModelForCausalLM.from_pretrained(name, dtype="auto", device_map="cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(name)
texts = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train")["text"]

frame = lf.LexFrame(model, tokenizer).init(texts)
frame

## Read, ablate and steer

`state` gives the hidden states of every layer at every position; their `logits` show what each state carries by itself. The logits at the last position show where the answer appears; `ablate` removes what a layer wrote, which shows who wrote the answer; `steer` moves the run at one layer, and the move is carried forward to the model's output.

In [ ]:
text = "The Eiffel Tower is located in the city of"
s = frame.state(text)
top = s[:, -1].logits().softmax(-1).topk(5)  # [layer, 5]: what every layer's state carries by itself
labels = [[tokenizer.decode([int(i)]) for i in row] for row in top.indices]
for layer, row in enumerate(labels):
    print(layer, row)

In [ ]:
import numpy as np

tops = [row[0] == " Paris" for row in labels]
print("Paris tops the logits from layer", tops.index(True) if any(tops) else -1)
paris = s[:, -1].logits().softmax(-1)[:, tokenizer.encode(" Paris", add_special_tokens=False)[0]]
print("the probability of Paris per layer:", paris.cpu().numpy().round(3))
base = float(s[-1, -1].logits([" Paris"])[0])
removed = [base - float(s[k, -1].ablate()[-1, -1].logits([" Paris"])[0]) for k in range(1, frame.num_layers + 1)]
print("the layers that wrote the prediction most:", np.argsort(removed)[::-1][:3])  # removing their write costs Paris most


In [ ]:
moved = s[16, -1].steer({" Paris": -4, " London": 4})  # the run moved at layer 16, carried forward to the output
name = lambda z: [tokenizer.decode([int(i)]) for i in z.topk(5).indices]
print("before:", name(s[-1, -1].logits()))
print("after: ", name(moved[-1, -1].logits()))
print("change:", (moved[-1, -1].logits([" Paris", " London"]) - s[-1, -1].logits([" Paris", " London"])).tolist())


## Compare with two baselines

The baselines live apart from the frame, in `lexframe.baselines`. The logit lens reads every layer with the output head; the Jacobian lens fits one linear map per layer. Both read every state of a layer alike, whatever its context. Fitting the Jacobian lens takes one backward pass per 8 hidden dimensions and sequence, which adds up on a free GPU: four sequences take several minutes here, and more sequences fit the lens better.

In [ ]:
from lexframe.baselines import JacobianLens, LogitLens

lenses = {"logit lens": LogitLens(model, tokenizer), "Jacobian lens": JacobianLens(model, tokenizer).fit(texts[:1000], size=4)}

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

words = [tokenizer.decode([i]) for i in tokenizer.encode(text, add_special_tokens=False)]
fig, axes = plt.subplots(1, 3, figsize=(18, 8), sharey=True, layout="constrained")
for ax, (title, method) in zip(axes, {**lenses, "LexFrame": frame}.items()):
    p, ids = method.state(text).logits().float().softmax(-1).max(-1)
    p, ids = p.cpu().numpy(), ids.cpu().numpy()
    ax.imshow(p, aspect="auto", origin="lower", cmap="Blues", vmin=0, vmax=1)
    for (layer, position), i in np.ndenumerate(ids):
        ax.text(position, layer, repr(tokenizer.decode([int(i)]))[1:-1][:10], ha="center", va="center", fontsize=5)
    ax.set_xticks(range(len(words)), words, rotation=45, ha="right")
    ax.set_title(title)
axes[0].set_ylabel("layer (0 is the embedding, the last is the output)")
plt.show()


## More

[docs/USAGE.md](https://github.com/jianguda/lexframe/blob/main/docs/USAGE.md) lists every command, [examples/walkthrough.py](https://github.com/jianguda/lexframe/blob/main/examples/walkthrough.py) goes through them, [examples/baselines.py](https://github.com/jianguda/lexframe/blob/main/examples/baselines.py) plots how far each method's logits lie from the model's output, and [examples/visualize.py](https://github.com/jianguda/lexframe/blob/main/examples/visualize.py) plots every layer's logits, where the state sits among the semantic bases, and what a steer does.